# Empirical Study of the Online Extra Trees (OXT) Regressor

Assignment devoloped by António Romão (up202108704) and Bruno Moreira (up202107143) for the Data Stream Mining course at FEUP.


## Table of Contents

1. [Introduction and Objectives](#1-introduction-and-objectives)
    - 1.1. [Context and Problem Statement](#11-context-and-problem-statement)
    - 1.2. [Assignment Objectives](#12-assignment-objectives)
2. [Instructions for Reproducibility](#2-instructions-for-reproducibility)
3. [Algorithms Overview: OXT and Baseline Regressors](#3-algorithms-overview-oxt-and-baseline-regressors)
    - 3.1. [Online Extra Trees (OXT)](#31-online-extra-trees-oxt)
    - 3.2. [Baseline Models for Comparison](#32-baseline-models-for-comparison)
4. [Datasets Specification and Sources](#4-datasets-specification-and-sources)
5. [Parameter and Hyperparameter Settings](#5-parameter-and-hyperparameter-settings)
6. [Empirical Evaluation Execution](#6-empirical-evaluation-execution)
    - 6.1. [Core Evaluator](#61-core-evaluator)
    - 6.2. [Hyperparameter Search](#62-hyperparameter-search)
7. [Results Analysis and Discussion](#7-results-analysis-and-discussion)
8. [Strengths, Limitations, and Conclusions](#8-strengths-limitations-and-conclusions)
9. [References](#9-references)

## 1. Introduction and Objectives

[[go back to top]](#table-of-contents)

This section establishes the foundation for our empirical study of the Online Extra Trees (OXT) regressor. Conducted as part of the Data Stream Mining course's first practical assignment, this study evaluates the OXT algorithm's ability to handle continuous data streams. Before detailing the experimental setup and results, we first define the core problem context and outline the primary objectives guiding our comparative analysis.

### 1.1. Context and Problem Statement

[[go back to section]](#1-introduction-and-objectives)

In traditional machine learning, models are typically trained on static, bounded datasets where multiple passes over the data are possible. However, in many modern applications, data is generated continuously and must be processed sequentially. This paradigm, known as Data Stream Mining, introduces significant challenges: algorithms must learn incrementally (usually in a single pass), operate under strict memory and computational time constraints, and continuously adapt to potential shifts in the underlying data distribution, known as concept drift.

This empirical study focuses on the specific problem of regression within a data stream setting. To address this, we evaluate the Online Extra Trees (OXT) regressor. The OXT algorithm is an ensemble learning method adapted for streaming data, designed to incrementally update a forest of randomized decision trees as new continuous target variables arrive. By analyzing OXT, we aim to understand how ensemble methods can effectively maintain predictive performance while adhering to the strict resource limitations of streaming environments.

### 1.2. Assignment Objectives

[[go back to section]](#1-introduction-and-objectives)

The overarching goal of this assigment is to perform a comprehensive empirical analysis of a specific data stream algorithm: in our case, the Online Extra Trees (OXT) regressor.

To achieve this, our study is guided by the following core objectives:
* **Understand the Algorithm:** Thoroughly investigate the OXT regressor to understand its underlying mechanics, state management, and theoretical assumptions in a streaming context.
* **Design a Reproducible Evaluation:** Develop and execute an informative and fully reproducible empirical evaluation comparing the OXT regressor against a chosen baseline comparison algorithm, utilizing at least two distinct datasets.
* **Critically Analyze Results:** Evaluate the experimental outcomes to identify the specific strengths and limitations of the OXT algorithm under varying streaming conditions.

## 2. Instructions for Reproducibility

[[go back to top]](#table-of-contents)

To ensure full reproducibility of this empirical evaluation, please set up an isolated virtual environment. This project was developed and tested using **Python 3.11**.

**Step-by-Step Guide:**
1. **Navigate to the directory:** Open your terminal and travel to the root of this project.
2. **Create the virtual environment:** Run `python3 -m venv dsm_env`
3. **Activate the environment:**
   * On macOS/Linux: `source dsm_env/bin/activate`
   * On Windows: `dsm_env\Scripts\activate`
4. **Install dependencies:** Run `pip install -r requirements.txt`
5. **Register the Jupyter Kernel:** Run `python -m ipykernel install --user --name=dsm_env --display-name "Python (DSM OXT)"`
6. **Execute:** Open this notebook, ensure the "Python (DSM OXT)" kernel is selected, and run all cells sequentially.

## 3. Algorithms Overview: OXT and Baseline Regressors

[[go back to top]](#table-of-contents)

This section outlines the theoretical foundations of the proposed Online Extra Trees algorithm and the selected baseline models. To rigorously evaluate the performance of OXT, we benchmark it against two established regressors: the Adaptive Random Forest (ARF) Regressor, serving as the "Gold Standard" state-of-the-art ensemble baseline, and the Hoeffding Tree Regressor (HTR), serving as the structural baseline for incremental tree learning.

### 3.1. Online Extra Trees (OXT)

[[go back to section]](#3-algorithms-overview-oxt-and-baseline-regressors)

The Online Extra Trees (OXT) algorithm is an adaptation of the batch Extremely Randomized Trees algorithm designed specifically for data stream mining. Traditional streaming tree ensembles compute the Hoeffding bound for every feature at every node to determine the optimal split, which is highly computationally expensive. OXT circumvents this bottleneck by strongly randomizing both the attribute selection and the split-point cutoffs. By evaluating only a random subset of features and a random split value for each, OXT drastically reduces the mathematical overhead required to grow the tree.

**State Management and Updates:** To operate in a single-pass streaming setting, OXT must incrementally maintain "state" at the leaf nodes rather than storing historical data. For continuous regression targets, each leaf tracks sufficient statistics—specifically, the instance count or sum of weights ($N$), the sum of target values ($\sum y$), and the sum of squared target values ($\sum y^2$). When a new instance arrives, it traverses the tree and updates these statistics at the corresponding leaf. Because OXT uses randomized split-points, it only tracks variance reduction metrics for a random subset of feature-cutoffs, skipping the exhaustive calculations required by traditional Hoeffding bounds to trigger a node split.

**Concept Drift Handling:**
To adapt to changing underlying data distributions, OXT actively monitors its predictive performance. Drift detectors (such as ADWIN) track the sequential error rates of the individual trees in the ensemble. When ADWIN detects a statistically significant increase in error (indicating concept drift), the obsolete tree is reset or replaced. This localized, tree-level adaptation allows the ensemble to seamlessly update its knowledge base without interrupting the stream processing.

**Hypothesis:** The primary hypothesis of this empirical evaluation is that OXT can leverage the ensemble effect to maintain high predictive accuracy while offering significant reductions in execution time and memory consumption compared to heavier streaming ensembles.

### 3.2. Baseline Models for Comparison

[[go back to section]](#3-algorithms-overview-oxt-and-baseline-regressors)

To contextualize the performance of OXT, it is evaluated against two distinct types of streaming regressors:

**The "Gold Standard" Ensemble: Adaptive Random Forest (ARF) Regressor**
* **Mechanism:** ARF is widely considered the state-of-the-art ensemble method for data streams. It adapts the traditional Random Forest by using online bagging (via Poisson distribution) to simulate bootstrapping. Crucially, ARF incorporates explicit drift detection (commonly ADWIN) to monitor the error rate of individual trees. When drift is detected, it trains "background trees" to seamlessly replace obsolete trees without interrupting the stream.
* **Rationale for Comparison:** ARF represents the ceiling for predictive performance in streaming ensembles. Comparing OXT to ARF isolates the trade-off between accuracy and resource efficiency. OXT must demonstrate that it can approximate or match ARF's error metrics (like MAE or RMSE) while proving superior in terms of memory footprint and processing speed.

**The Structural Baseline: Hoeffding Tree Regressor (HTR)**
* **Mechanism:** Also known in literature as the Fast Incremental Model Tree with Drift Detection (FIMT-DD), the Hoeffding Tree is the foundational single-tree architecture for online learning. It processes instances one by one, using the statistical Hoeffding bound to guarantee, with high probability, that a split made on a small sample of data is the exact same split that would be made if the tree had access to infinite data.
* **Rationale for Comparison:** HTR represents the absolute performance floor that an ensemble model must exceed. Because an ensemble introduces inherent computational complexity by managing multiple trees, it must yield a statistically significant improvement in predictive accuracy over a single, well-optimized Hoeffding Tree. If OXT cannot consistently outperform the HTR, the overhead of the ensemble architecture is not justified.

In [2]:
from river import forest
from river import tree
from river import drift
from river import metrics

oxt_example = forest.OXTRegressor(
    n_models=10,                              # number of trees used -> more trees -> more accuracy (plateaus)
    max_features="sqrt",                      # number of features evaluated at each split
    resampling_strategy="subbagging",         # how instances are passed to trees
    resampling_rate=0.5,                      # subbaging only, prob. of incoming instance being used by a tree
    detection_mode="all",                     # controls drift adaptation
    warning_detector=drift.ADWIN(delta=0.01), # statistical algorithms used to monitor error
    drift_detector=drift.ADWIN(delta=0.001),  # on ADWIN (standard) lower delta -> stricter threshold for drift
    max_depth=None,                           # limit tree growth to constraint memory
    randomize_tree_depth=False,               # unique to OXT, forces diff. depths across trees for diversity
    track_metric=metrics.MAE(),               # metric used to evaluate tree performance
    disable_weighted_vote=False               # have prediction weights based on internal tree metrics or just average
)

arf_example = forest.ARFRegressor(
    n_models=10,                              # same as above
    max_features="sqrt",                      # same as above
    aggregation_method="mean",                # how ensemble combine predictions
    warning_detector=drift.ADWIN(delta=0.01), # same as above
    drift_detector=drift.ADWIN(delta=0.001),  # same as above
    grace_period=50,                          # how many instances a leaf must observe before bound calculation for split
    delta=1e-7,                               # target for Hoeffding bound calculations
    tau=0.05,                                 # between two best features, split anyway if difference falls below threshold
    leaf_prediction="adaptive",               # how leaves make predictions
    seed=42                                   # random start seed
)

htr_example = tree.HoeffdingTreeRegressor(
    grace_period=200,                         # ensure solid calculation before split
    max_depth=None,                           # same as above
    delta=1e-7,                               # same as in ARF
    tau=0.05,                                 # same as in ARF
    leaf_prediction="adaptive",               # same as in ARF
    nominal_attributes=None                   # when you have label-encoded features, pass indices here
)

## 4. Datasets Specification and Sources

[[go back to top]](#table-of-contents)

To comprehensively evaluate the OXT regressor, we utilize one synthetic and one real-world dataset, both designed for continuous regression tasks. Both datasets are sourced and loaded via the `river.datasets` API.

**1. Synthetic Data: The Friedman Generator**
We use the Friedman Generator (`river.datasets.synth.Friedman`) to simulate a complex, non-linear mathematical relationship between 10 features and a continuous target. Generating 100,000 instances allows for full experimental control and the explicit injection of concept drift to precisely measure the algorithms' adaptation and recovery mechanisms. We configure three distinct stream variations:
* **Stationary:** The baseline mathematical function remains constant.
* **Abrupt Drift:** At instance 50,000, the underlying function instantly changes by swapping the mathematical roles of specific features ($x_0$ with $x_4$, and $x_1$ with $x_3$).
* **Gradual Drift:** Using a sigmoid function centered around instance 50,000, the stream smoothly transitions between the original and the swapped-feature concepts over a window of 10,000 instances.

**2. Real-World Data: Bike Sharing**
For organic data, we use the Bike Sharing dataset (`river.datasets.Bikes`), originally sourced from the UCI Machine Learning Repository/OpenML. This dataset contains 182,470 instances and 8 features, recording the hourly count of rented bicycles in a city-share system alongside meteorological and temporal data. It is a highly cited data stream benchmark because it naturally embeds continuous environmental dynamics—such as recurring daily behaviors and gradual seasonal weather changes. It combines a manageable size for rapid iteration with sufficient feature richness to be deeply informative.

In [3]:
from river import datasets
from river.datasets import synth

import math
import random

# Synthetic: Friedman Generator

# -> generate and yield instance per call
def load_friedman_stationary(n_instances: int = 100_000, seed: int = 42):
    gen = synth.Friedman(seed=seed)

    for i, (x, y) in enumerate(gen):
        if i >= n_instances:
            print("DATA EXHAUSTED: FINISHED LOADING")
            break
        yield x, y

# same as above, with an abrupt drift
def load_friedman_with_abrupt_drift(n_instances: int = 100_000, drift_position: int = 50_000, seed: int = 42):
    rng = random.Random(seed)

    for i in range(n_instances):
        # 10 uniform features in [0, 1]
        features = {f"x{j}": rng.uniform(0, 1) for j in range(10)}

        noise = rng.gauss(0, 1)

        if i < drift_position:
            # original function for friedman
            y = (
                10 * math.sin(math.pi * features["x0"] * features["x1"])
                + 20 * (features["x2"] - 0.5) ** 2
                + 10 * features["x3"]
                + 5 * features["x4"]
                + noise
            )
        else:
            # drifted function, swap roles (x0↔x4, x1↔x3)
            y = (
                10 * math.sin(math.pi * features["x4"] * features["x3"])
                + 20 * (features["x2"] - 0.5) ** 2
                + 10 * features["x1"]
                + 5 * features["x0"]
                + noise
            )

        yield features, y

# same as above, with gradual drift
def load_friedman_with_gradual_drift(n_instances: int = 100_000, drift_center: int = 50_000, drift_width: int = 10_000, seed: int = 42):
    rng = random.Random(seed)

    # controls sigmoid function, scaling
    # to be balanced with drift_window
    steepness = 12.0 / max(drift_width, 1)

    for i in range(n_instances):
        # 10 uniform features in [0, 1]
        features = {f"x{j}": rng.uniform(0, 1) for j in range(10)}
        noise = rng.gauss(0, 1)
 
        # original Friedman function
        y_a = (
            10 * math.sin(math.pi * features["x0"] * features["x1"])
            + 20 * (features["x2"] - 0.5) ** 2
            + 10 * features["x3"]
            + 5 * features["x4"]
            + noise
        )
 
        # swapped feature roles (x0↔x4, x1↔x3)
        y_b = (
            10 * math.sin(math.pi * features["x4"] * features["x3"])
            + 20 * (features["x2"] - 0.5) ** 2
            + 10 * features["x1"]
            + 5 * features["x0"]
            + noise
        )
 
        # sigmoid blending of two concepts
        p_new = 1.0 / (1.0 + math.exp(-steepness * (i - drift_center)))
 
        # blend the two targets smoothly
        y = (1 - p_new) * y_a + p_new * y_b
 
        yield features, y

# Real: Bike Sharing

# -> dataset gets loaded and cached
# -> each call yields an instance
def load_bikes():
    dataset = datasets.Bikes()
    for x, y in dataset:
        yield x, y

# return metadata about the dataset
def get_bikes_info():
    dataset = datasets.Bikes()
    return {
        "name": dataset.__class__.__name__,
        "n_instances": dataset.n_samples,
        "n_features": dataset.n_features,
        "task": str(dataset.task),
    }


## 5. Parameter and Hyperparameter Settings

[[go back to top]](#table-of-contents)

To ensure a fair algorithmic comparison between OXT and its baselines, we define a dedicated search grid for each algorithm (symmetric tuning, parameters get mirrored whenever shared) rather than only one for the proposed method. We also provide the same tuning budget (random-search configuration count) to each model. Specifically, we allocate a budget of 50 random search iterations per model.

OXT's grid spans the parameters that define its distinctive behaviour. ARF's grid mirrors OXT's wherever the parameters are shared, and adds `lambda_value` to cover Oza's online bagging versus Leveraging Bagging. HTR's grid is naturally smaller because HTR has fewer degrees of freedom, but it is tuned with the same random-search budget, which produces denser coverage of its (smaller) space.

A few parameters are left out, for principled reasons. OXT's `detection_mode` and the ADWIN $\delta$ values are fixed across configurations because the tuning stream is stationary; no drift signal exists there for the search to exploit. The same logic applies to ARF's drift and warning detectors. OXT's `randomize_tree_depth` is fixed to `False` to keep `max_depth` comparable across ensembles, and `disable_weighted_vote` is fixed to match ARF's `aggregation_method="mean"` so that differences in aggregation do not contaminate the ensemble-to-ensemble comparison.

In [4]:
OXT_GRID = {
    "n_models": [5, 10, 25, 50],
    "max_features": ["sqrt", "log2", 0.5, None],
    "resampling_strategy": ["subbagging", "bagging"],
    "resampling_rate": [0.5, 0.75, 1],
    "leaf_prediction": ["mean", "model", "adaptive"],
    "grace_period": [50, 100, 200],
    "delta": [1e-7, 1e-3, 0.01],
    "max_depth": [None, 10, 20],
}

ARF_GRID = {
    "n_models": [5, 10, 25, 50],
    "max_features": ["sqrt", "log2", 0.5, None],
    "lambda_value": [1, 6],
    "leaf_prediction": ["mean", "model", "adaptive"],
    "grace_period": [50, 100, 200],
    "delta": [1e-7, 1e-3, 0.01],
    "max_depth": [None, 10, 20],
}

HTR_GRID = {
    "grace_period": [50, 100, 200, 500],
    "max_depth": [None, 10, 20, 30],
    "delta": [1e-7, 1e-5, 1e-3, 0.01],
    "tau": [0.05, 0.1],
    "leaf_prediction": ["mean", "model", "adaptive"],
    "binary_split": [True, False],
}

GRIDS = {"OXT": OXT_GRID, "ARF": ARF_GRID, "HTR": HTR_GRID}

## 6. Empirical Evaluation Execution

[[go back to top]](#table-of-contents)

In this component, we use the Friedman-Stationary dataset to find each models best general-purpose configuration, to be evaluated on its own while it found the best parameters for each model, for said models then to be used in the three other use cases (Friedman-Abrupt, Friedman-Gradual and Bikes).

### 6.1. Core Evaluator

[[go back to section]](#6-empirical-evaluation-execution)

To rigorously evaluate the trade-offs between predictive accuracy and resource consumption, we developed a lightweight, custom prequential evaluator. This function incrementally trains the model while tracking cumulative error metrics, rolling window metrics, execution throughput, and the model's memory footprint in bytes.

In [5]:
import time
import pickle
from river import metrics, utils  # <-- Added utils import

# helper function to evaluate model size
def model_size_bytes(model) -> int:
    return len(pickle.dumps(model))

# test-then-train evaluation
def prequential_eval(model, stream, n_instances=100_000, window=1_000, track_memory_every=5_000):
    mae_cum, rmse_cum, r2_cum = metrics.MAE(), metrics.RMSE(), metrics.R2()
    
    # FIXED: Use utils.Rolling instead of metrics.Rolling
    mae_roll = utils.Rolling(metrics.MAE(), window_size=window)
    rmse_roll = utils.Rolling(metrics.RMSE(), window_size=window)
    
    history, memory_trace = [], []
    
    t0 = time.perf_counter() 
    for i, (x, y) in enumerate(stream):
        if i >= n_instances:
            break
            
        # Test
        y_hat = model.predict_one(x)
        if y_hat is None: 
            y_hat = 0.0
            
        # Update metrics 
        for m in (mae_cum, rmse_cum, r2_cum, mae_roll, rmse_roll):
            m.update(y, y_hat)
            
        # Train
        model.learn_one(x, y)

        # Track history per window
        if (i + 1) % window == 0:
            history.append({
                "step": i + 1,
                "mae_rolling": mae_roll.get(),
                "rmse_rolling": rmse_roll.get(),
                "mae_cum": mae_cum.get(),
                "rmse_cum": rmse_cum.get(),
            })
            
        # Track memory
        if (i + 1) % track_memory_every == 0:
            memory_trace.append((i + 1, model_size_bytes(model)))

    total_time = time.perf_counter() - t0
    
    return {
        "mae": mae_cum.get(),
        "rmse": rmse_cum.get(),
        "r2": r2_cum.get(),
        "history": history,
        "memory_trace": memory_trace,
        "throughput": n_instances / total_time,
        "total_time_s": total_time,
        "final_mem_bytes": model_size_bytes(model),
    }

### 6.2. Hyperparameter Search

[[go back to section]](#6-empirical-evaluation-execution)

To balance computational cost with finding an ideal configuration, we perform a random search. We run this over 20,000 instances of the stationary Friedman stream to find generalizable hyperparameters.

> **Note on Reproducibility:** The following tuning block performs a random search over 50 configurations across 3 algorithms. Depending on your hardware, this cell may take several minutes to execute.

In [6]:
import pandas as pd
import random
from itertools import product
from copy import deepcopy
from river import drift, forest, metrics, tree
from tqdm.auto import tqdm  # <-- Added tqdm import

# grid search parameters
SEEDS = [42, 777, 1234]
N_INSTANCES = 100_000
TUNING_PREFIX = 20_000
N_CONFIGS = 50
TUNING_SEED = 999
ROLLING_WINDOW = 1_000
WINDOW_SIZE = 500

# uniformly sample n configs from the cartesian product of grid
def sample_configs(grid: dict, n: int, seed: int = 0):
    rng = random.Random(seed)
    all_keys = list(grid.keys())
    all_combos = list(product(*[grid[k] for k in all_keys]))
    rng.shuffle(all_combos)
    return [dict(zip(all_keys, combo)) for combo in all_combos[:n]]

# automatically build model based on algorithm, configuration and seed
def build_model(algo: str, cfg: dict, seed: int):
    if algo == "OXT":
        # Prevent River crash: 'bagging' requires an integer rate >= 1
        if cfg.get("resampling_strategy") == "bagging" and cfg.get("resampling_rate", 1) < 1:
            cfg["resampling_rate"] = 1
            
        return forest.OXTRegressor(
            **cfg,
            detection_mode="all",
            warning_detector=drift.ADWIN(delta=0.01),
            drift_detector=drift.ADWIN(delta=0.001),
            seed=seed
        )
        
    if algo == "ARF":
        return forest.ARFRegressor(
            **cfg,
            aggregation_method="mean",
            warning_detector=drift.ADWIN(delta=0.01),
            drift_detector=drift.ADWIN(delta=0.001),
            tau=0.05,
            seed=seed,
        )
        
    if algo == "HTR":
        return tree.HoeffdingTreeRegressor(**cfg)
        
    raise ValueError(algo)

def tune(algo: str, tuning_stream_factory, n_configs=N_CONFIGS, seed=TUNING_SEED, selection="rmse"):
    configs = sample_configs(GRIDS[algo], n_configs, seed=seed)
    records = []
    
    # FIXED: Wrap configs in tqdm for a progress bar
    pbar = tqdm(configs, desc=f"Tuning {algo}")
    
    for cfg in pbar:
        model = build_model(algo, cfg, seed=seed)
        out = prequential_eval(model, tuning_stream_factory(), n_instances=TUNING_PREFIX, window=WINDOW_SIZE)
        
        records.append({
            **cfg,
            "mae": out["mae"],
            "rmse": out["rmse"],
            "r2": out["r2"],
            "throughput": out["throughput"],
            "mem_kb": out["final_mem_bytes"] / 1024,
            "time_s": out["total_time_s"],
            "history": out["history"],
            "memory_trace": out["memory_trace"],
        })
        
        # FIXED: Update progress bar with current metrics instead of printing a new line
        pbar.set_postfix({
            "RMSE": f"{out['rmse']:.3f}", 
            "MAE": f"{out['mae']:.3f}", 
            "thr": f"{out['throughput']:.0f}/s", 
            "mem": f"{out['final_mem_bytes']/1024:.0f}KB"
        })
              
    df = pd.DataFrame(records)
    
    # depending on selection, prioritize rmse, or pareto score (rmse/mem/throughput)
    if selection == "rmse":
        df_ranked = df.sort_values("rmse").reset_index(drop=True)
    elif selection == "pareto":
        eps = 1e-12
        def norm(s, higher_is_better=False):
            lo, hi = s.min(), s.max()
            z = (s - lo) / (hi - lo + eps)
            return 1 - z if higher_is_better else z
            
        df = df.assign(
            _rmse_n = norm(df["rmse"]),
            _thr_n = norm(df["throughput"], higher_is_better=True),
            _mem_n = norm(df["mem_kb"]),
        )
        # weigh accuracy most, but don't ignore the other two (tunable)
        df["score"] = 0.6 * df["_rmse_n"] + 0.2 * df["_thr_n"] + 0.2 * df["_mem_n"]
        df_ranked = df.sort_values("score").reset_index(drop=True)
    else:
        raise ValueError(f"Unknown selection={selection!r}")
        
    # get best config based on top of table
    best_cfg = df_ranked.iloc[0].to_dict()
    for k in ("mae", "rmse", "r2", "throughput", "mem_kb", "time_s", "history", "memory_trace", "score", "_rmse_n", "_thr_n", "_mem_n"):
        best_cfg.pop(k, None)
        
    return best_cfg, df_ranked

# USAGE
tuning_factory = lambda: load_friedman_stationary(TUNING_PREFIX, seed=TUNING_SEED)

# ensembles: select on the composite Pareto score (respects OXT's tradeoff claim)
best_oxt, oxt_search = tune("OXT", tuning_factory, selection="pareto")
best_arf, arf_search = tune("ARF", tuning_factory, selection="pareto")

# single tree: no tradeoff story to respect, RMSE-only is fine
best_htr, htr_search = tune("HTR", tuning_factory, selection="rmse")

BEST_CFGS = {"OXT": best_oxt, "ARF": best_arf, "HTR": best_htr}

print("\n=== Selected configurations ===")
for algo, cfg in BEST_CFGS.items():
    print(f"{algo}: {cfg}")

Tuning OXT:   0%|          | 0/50 [00:00<?, ?it/s]

DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAU

Tuning ARF:   0%|          | 0/50 [00:00<?, ?it/s]

DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAU

Tuning HTR:   0%|          | 0/50 [00:00<?, ?it/s]

DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAUSTED: FINISHED LOADING
DATA EXHAU

**Save the results:**

In [7]:
import pickle
import os

# Create a directory to store your results safely
os.makedirs("checkpoints", exist_ok=True)

# Bundle everything into a single dictionary
tuning_artifacts = {
    "BEST_CFGS": BEST_CFGS,
    "oxt_search": oxt_search,
    "arf_search": arf_search,
    "htr_search": htr_search
}

# Serialize and save to disk
with open("checkpoints/tuning_results.pkl", "wb") as f:
    pickle.dump(tuning_artifacts, f)
    
print("✅ Tuning results successfully saved to 'checkpoints/tuning_results.pkl'")

✅ Tuning results successfully saved to 'checkpoints/tuning_results.pkl'


**Load the results:**

In [8]:
import pickle
import os

checkpoint_path = "checkpoints/tuning_results.pkl"

if os.path.exists(checkpoint_path):
    with open(checkpoint_path, "rb") as f:
        tuning_artifacts = pickle.load(f)
        
    BEST_CFGS = tuning_artifacts["BEST_CFGS"]
    oxt_search = tuning_artifacts["oxt_search"]
    arf_search = tuning_artifacts["arf_search"]
    htr_search = tuning_artifacts["htr_search"]
    
    print("✅ Tuning results successfully loaded from disk!\n")
    print("=== Loaded Configurations ===")
    for algo, cfg in BEST_CFGS.items():
        print(f"{algo}: {cfg}")
else:
    print(f"⚠️ No checkpoint found at '{checkpoint_path}'. You need to run the tuning block first.")

✅ Tuning results successfully loaded from disk!

=== Loaded Configurations ===
OXT: {'n_models': 5, 'max_features': 'sqrt', 'resampling_strategy': 'bagging', 'resampling_rate': 1.0, 'leaf_prediction': 'adaptive', 'grace_period': 50, 'delta': 1e-07, 'max_depth': 10.0}
ARF: {'n_models': 5, 'max_features': 'sqrt', 'lambda_value': 6, 'leaf_prediction': 'model', 'grace_period': 200, 'delta': 0.01, 'max_depth': 20.0}
HTR: {'grace_period': 100, 'max_depth': 10.0, 'delta': 0.01, 'tau': 0.1, 'leaf_prediction': 'adaptive', 'binary_split': False}


### 6.3. Tuning Observations

[[go back to section]](#6-empirical-evaluation-execution)

The hyperparameter search successfully identified optimal configurations for our three models. Notably, the Pareto-style selection mechanism chose **5 models (trees)** for both the OXT and ARF ensembles—the lowest available option in our search grid. 

This directly highlights the realities of Data Stream Mining: while larger forests (e.g., 25 or 50 trees) might yield marginal improvements in absolute predictive accuracy, the strict penalties applied to memory consumption and execution throughput in our scoring function favored highly lightweight ensembles. This ensures that the models moving into the final evaluation phase are strictly optimized for the severe resource constraints of a streaming environment.

## 7. Results Analysis and Discussion

[[go back to top]](#table-of-contents)

## 8. Strengths, Limitations and Conclusions

[[go back to top]](#table-of-contents)

## 9. References

- "Online Extra Trees Regressor" (Mastelini et al., 2022)

- River Documentation (riverml.xyz)

- OpenML (openml.org)

[[go back to top]](#table-of-contents)